In [ ]:
# Cell 1: Client Setup
import json
import boto3
import redis
import subprocess

# LocalStack SQS Client on port 4567
sqs = boto3.client(
    'sqs',
    endpoint_url='http://127.0.0.1:4567',
    region_name='us-east-1',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)
queue_url = "http://127.0.0.1:4567/000000000000/task-queue"

# Redis Cache Client
cache = redis.Redis(host='127.0.0.1', port=6379, decode_responses=True)

# Replace with the file URL that was already successfully processed in your worker terminal
TEST_FILE_URL = "s3://lta-traffic-raw/lta_traffic_198580c7-30df-4245-acf2-d3cf76873998.json"

print("Clients initialized. Target file:", TEST_FILE_URL)

In [ ]:
# Cell 2: Test Dynamic Duplicate Filtering (At-Most-Once)
# 1. Verify the URL is currently remembered in Redis history
in_cache = cache.exists(TEST_FILE_URL)
print(f"Is file URL recorded in Redis history? -> {bool(in_cache)}")

if not in_cache:
    # If not recorded yet, seed it so the filter has history to check against
    cache.set(TEST_FILE_URL, "completed", ex=86400)
    print("Seeded file URL into Redis history.")

# 2. Re-inject the exact same task into SQS
duplicate_payload = {"file_url": TEST_FILE_URL}
res = sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(duplicate_payload))

print(f"\n[SENT] Retransmitted task injected to SQS (MessageId: {res['MessageId']})")
print("\n--- WHAT TO VERIFY IN WORKER TERMINAL ---")
print("Look at 'docker logs -f local-env-worker-1':")
print("1. It must print: 'Duplicate task detected: ... Discarding to prevent redundant compute.'")
print("2. It must NOT print 'Downloading ... from S3' or 'Writing clean data to PostgreSQL'.")

In [ ]:
# Cell 3: Test Database Idempotency (At-Least-Once Safety Net)
# 1. Wipe Redis history (simulates cache crash/loss of history)
cache.flushall()
print("Redis history wiped. Dynamic filter is now bypassed.")

# 2. Re-inject the exact same task into SQS
duplicate_payload = {"file_url": TEST_FILE_URL}
res = sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(duplicate_payload))

print(f"\n[SENT] Retransmitted task injected without cache history (MessageId: {res['MessageId']})")
print("\n--- WHAT TO VERIFY IN WORKER TERMINAL ---")
print("Look at 'docker logs -f local-env-worker-1':")
print("1. It downloads the file from S3 (filter let it through).")
print("2. It prints: 'Successfully saved vehicle SG-6216 to database!'")
print("3. Notice: No 'UniqueViolation' crash occurred because 'ON CONFLICT DO UPDATE' handled it.")

In [ ]:
# Cell 4: Verify Database State
check_sql = "SELECT vehicle_id, COUNT(*) FROM cleansed_traffic GROUP BY vehicle_id HAVING COUNT(*) > 1;"

cmd = [
    "docker", "exec", "-i", "local-env-postgres-1",
    "psql", "-U", "admin", "-d", "dataclean_db",
    "-c", check_sql
]

result = subprocess.run(cmd, capture_output=True, text=True)
print("Query Output:")
print(result.stdout)

if "(0 rows)" in result.stdout:
    print("RESULT: PASSED. 0 duplicate records found. Idempotent UPSERT verified.")
else:
    print("RESULT: FAILED. Duplicate records were detected in the database.")